In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sklearn
import torch.nn as nn
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

Task 1

In [2]:
df = pd.read_csv("movies.csv")

df = df[["overview", "tagline", "keywords", "genres", "vote_average"]]
df = df.dropna(subset=["overview", "tagline", "keywords"], how="all")

df.head()

,overview,tagline,keywords,genres,vote_average
0,"In the 22nd century, a paraplegic Marine is di...",Enter the World of Pandora.,culture clash future space war space colony so...,Action Adventure Fantasy Science Fiction,7.2
1,"Captain Barbossa, long believed to be dead, ha...","At the end of the world, the adventure begins.",ocean drug abuse exotic island east india trad...,Adventure Fantasy Action,6.9
2,A cryptic message from Bond’s past sends him o...,A Plan No One Escapes,spy based on novel secret agent sequel mi6,Action Adventure Crime,6.3
3,Following the death of District Attorney Harve...,The Legend Ends,dc comics crime fighter terrorist secret ident...,Action Crime Drama Thriller,7.6
4,"John Carter is a war-weary, former military ca...","Lost in our world, found in another.",based on novel mars medallion space travel pri...,Action Adventure Science Fiction,6.1


In [3]:
import re

for col in ["overview", "tagline", "keywords"]:
    df[col] = df[col].fillna("").str.lower()
    df[col] = df[col].str.replace(r"http\S+|www\S+", "", regex=True)  # remove URLs
    df[col] = df[col].str.replace(r"[^a-z\s]", " ", regex=True)                # remove punctuation & numbers
    df[col] = df[col].str.replace(r"\s+", " ", regex=True).str.strip()       # remove extra whitespace

In [4]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    shuffle=True,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    shuffle=True,
)

print(len(train_df), len(val_df), len(test_df))

3361 720 721


Task_3

In [5]:
import numpy as np

glove_embeddings = {}
embedding_dim = 200

with open("glove.2024.wikigiga.200d/glove.2024.wikigiga.200d.txt", "r", encoding="utf-8") as f:
    for line in f:
        values = line.strip().split()
        
        # skip invalid lines
        if len(values) != embedding_dim + 1:
            continue
        
        word = values[0]
        
        try:
            vector = np.asarray(values[1:], dtype="float32")
            glove_embeddings[word] = vector
        except:
            continue

print("Loaded embeddings:", len(glove_embeddings))

Loaded embeddings: 1287614


In [6]:
glove_vocab = set(glove_embeddings.keys())
print(len(glove_vocab))

1287614


In [7]:
dataset_vocab = set()

for text in df["overview"]:
    dataset_vocab.update(text.split())

covered = sum(1 for word in dataset_vocab if word in glove_vocab)
coverage = covered / len(dataset_vocab) * 100

print(f"Coverage: {coverage:.2f}%")

Coverage: 98.92%


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    vocabulary=glove_vocab
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df["overview"])

/home/tanmay/Work/IT549/IT549_Labs/.venv-1/lib/python3.14/site-packages/sklearn/feature_extraction/text.py:1378: UserWarning: Upper case characters found in vocabulary while 'lowercase' is True. These entries will not be matched with any documents
  warnings.warn(


In [9]:
def doc_embedding(text, tfidf_vectorizer, feature_names):
    vec = np.zeros(embedding_dim)

    row = tfidf_vectorizer.transform([text])
    indices = row.indices
    weights = row.data

    weight_sum = 0

    for idx, weight in zip(indices, weights):
        word = feature_names[idx]

        if word in glove_embeddings:
            vec += weight * glove_embeddings[word]
            weight_sum += weight

    if weight_sum > 0:
        vec /= weight_sum

    return vec

In [ ]:
feature_names = tfidf_vectorizer.get_feature_names_out()

X_train_overview = np.array([
    doc_embedding(text, tfidf_vectorizer, feature_names)
    for text in train_df["overview"]
])

X_val_overview = np.array([
    doc_embedding(text, tfidf_vectorizer, feature_names)
    for text in val_df["overview"]
])

X_test_overview = np.array([
    doc_embedding(text, tfidf_vectorizer, feature_names)
    for text in test_df["overview"]
])

print(X_train_overview.shape, X_val_overview.shape, X_test_overview.shape)

(3361, 200) (720, 200) (721, 200)


Task 3

In [17]:
from sklearn.metrics import mean_squared_error
mean_rating = train_df["vote_average"] .mean()

baseline_preds = np.array([mean_rating] * len(test_df))
baseline_mse = mean_squared_error(test_df["vote_average"], baseline_preds)
baseline_rmse = np.sqrt(baseline_mse)

print("Baseline MSE:", baseline_mse)
print("Baseline RMSE:", baseline_rmse)

Baseline MSE: 1.4822112800241465
Baseline RMSE: 1.2174609973318022


In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RatingModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.net(x)

model = RatingModel(200)

In [ ]:
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)